#### Import des librairies

In [1]:
import pandas as pd
from tqdm import tqdm
import numpy as np

In [2]:
path_file = '~/Bureau/exports/20241112/AGS_20241112_exports_agronomes_-archive/AGS_20241112_exports_agronomes_assolees_synthetisees.csv'

ENTREPOT_PATH = '~/Bureau/utils/data/'
DIRODUR_FILES_PATH = '~/Bureau/projets/DIRODUR/magasin/'
df = {}

In [3]:
df['assolees_synthetisees'] = pd.read_csv(path_file, sep = '@')

#### Import des données

In [4]:
# -------------------------- #
# IMPORT DES DONNÉES DIRODUR #
# -------------------------- #
dirodur_files = ['correspondance_destination_gcpe']

for file in dirodur_files:
    df[file] = pd.read_csv(DIRODUR_FILES_PATH + file+'.csv', sep = ';')

In [5]:
# ----------------------------------------- #
# CRÉATION DU RÉFÉRENTIEL DE MATCH D'UNITÉS #
# ----------------------------------------- #
dict_unites = {
    'q/ha (humidité ramenée à la norme)' : 'Q_HA_TO_STANDARD_HUMIDITY',
    't MS/ha' : 'TONNE_MS_HA',
    't sucre/ha' : 'TONNE_SUGAR_HA',
    't/ha' : 'TONNE_HA',
    'tonne_racines_ha_16_pourc' : 'TONNE_RACINES_HA_16_POURC', 
    'q/ha' : 'Q_HA',
    'kg/m²' : 'KG_M2',
    'unité/ha' : 'UNITE_HA',
    'hl/ha' : 'HL_HA'
}
df['unite_rendement'] = pd.DataFrame.from_dict(dict_unites, orient='index', columns=['unite_agrosyst']).reset_index().rename(columns={'index':'unite_nl'})

In [6]:
# complétion du référentiel transmis par les agronomes
left = df['correspondance_destination_gcpe']
right = df['unite_rendement']
df['correspondance_destination_gcpe'] = pd.merge(left, right, left_on = 'Unité_rendement', right_on = 'unite_nl', how = 'left')

In [7]:
# ----------------------------- #
# IMPORT DES DONNÉES DATAGROSYST#
# ----------------------------- #


def import_df(df_name, path_data, sep, index_col=None):
    df[df_name] = pd.read_csv(path_data+df_name+'.csv', sep = sep, index_col=index_col, low_memory=False).replace({'\r\n': '\n'}, regex=True)

def import_dfs(df_names, path_data, sep = ',', index_col=None, verbose=False):
    for df_name in tqdm(df_names) : 
        if(verbose) :
            print(" - ", df_name)
        import_df(df_name, path_data, sep, index_col=index_col)

tables_with_id = [
    'recolte_rendement_prix', 
    'action_realise',
    'action_synthetise',
    'action_realise_agrege', 
    'action_synthetise_agrege',
    'destination_valorisation',
    'recolte_rendement_prix_restructure', 
    'composant_culture', 
    'espece', 
    'variete'
]

tables_without_id = [
    'sdc_magasin_dirodur', 
    'itk_magasin_dirodur'
]

# import des données de l'entrepôt avec la colonne 'id' en index 
import_dfs(tables_with_id, ENTREPOT_PATH, sep = ',', index_col='id', verbose=False)

# import des données du magasin
import_dfs(tables_without_id, ENTREPOT_PATH, sep = ',', verbose=False)

  0%|          | 0/10 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:05<00:00,  2.58s/it]


In [10]:
studied_ids = ['fr.inra.agrosyst.api.entities.action.HarvestingActionValorisation_8432e1a2-54d1-42d6-9c09-67d70b609526',
 'fr.inra.agrosyst.api.entities.action.HarvestingActionValorisation_5cf82905-16b5-4226-8950-04148b318949',
 'fr.inra.agrosyst.api.entities.action.HarvestingActionValorisation_705f458c-427d-4c46-950a-4ea7a9e51d6e',
 'fr.inra.agrosyst.api.entities.action.HarvestingActionValorisation_a66a0af9-a9d3-4579-b992-7ea5d50a114a',
 'fr.inra.agrosyst.api.entities.action.HarvestingActionValorisation_d5be281a-2864-48a8-9a14-21c62332df38',
 'fr.inra.agrosyst.api.entities.action.HarvestingActionValorisation_6eaf5102-6845-4927-bbd8-15bdea3a8c87',
 'fr.inra.agrosyst.api.entities.action.HarvestingActionValorisation_cb821414-9f76-4ade-829e-de5be7cd3368',
 'fr.inra.agrosyst.api.entities.action.HarvestingActionValorisation_917a7266-9e5a-4d42-94d2-f5ab016397bb',
 'fr.inra.agrosyst.api.entities.action.HarvestingActionValorisation_4e48ae97-b971-42e9-a284-89b5cdebe419',
 'fr.inra.agrosyst.api.entities.action.HarvestingActionValorisation_bbab7546-12e9-0b33-c6e5-0143dc11556c']

In [13]:
df['recolte_rendement_prix_test'] = df['recolte_rendement_prix'].loc[studied_ids]
df['action_synthetise_test'] = df['action_synthetise'].loc[
    df['action_synthetise'].index.isin(df['recolte_rendement_prix_test']['action_id'])
]
df['destination_valorisation_test'] = df['destination_valorisation'].loc[
    df['destination_valorisation'].index.isin(df['recolte_rendement_prix_test']['destination_id'])
]
df['action_synthetise_agrege_test'] = df['action_synthetise_agrege'].loc[
    df['action_synthetise_agrege'].index.isin(df['action_synthetise_test'].index)
]
df['recolte_rendement_prix_restructure_test']= df['recolte_rendement_prix_restructure'].loc[
    df['recolte_rendement_prix_restructure'].index.isin(df['recolte_rendement_prix_test'].index)
]
df['composant_culture_test'] = df['composant_culture'].loc[
    df['composant_culture'].index.isin(df['recolte_rendement_prix_restructure_test']['composant_culture_id'])
]
df['espece_test'] = df['espece'].loc[
    df['espece'].index.isin(df['composant_culture_test']['espece_id'])
]
df['variete_test'] = df['variete'].loc[
    df['variete'].index.isin(df['composant_culture_test']['variete_id'])
]

df['correspondance_destination_gcpe_test'] = df['correspondance_destination_gcpe'].loc[
    df['correspondance_destination_gcpe']['code_destination_A'].isin(df['destination_valorisation_test']['code_destination_a'])
]

In [14]:
path='./'
df['recolte_rendement_prix_test'].to_csv(path+'recolte_rendement_prix'+'.csv')
df['action_synthetise_test'].to_csv(path+'action_synthetise'+'.csv')
df['destination_valorisation_test'].to_csv(path+'destination_valorisation'+'.csv')
df['action_synthetise_agrege_test'].to_csv(path+'action_synthetise_agrege'+'.csv')
df['recolte_rendement_prix_restructure_test'].to_csv(path+'recolte_rendement_prix_restructure'+'.csv')
df['composant_culture_test'].to_csv('composant_culture'+'.csv')
df['correspondance_destination_gcpe_test'].to_csv('correspondance_destination_gcpe'+'.csv')
df['espece_test'].to_csv('espece'+'.csv')
df['variete_test'].to_csv('variete'+'.csv')